# 01 – Data Cleaning
IT Service Desk & SLA Analytics for fictional Pi Systems Private Limited using completely synthetic data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
raw=pd.read_csv(ROOT/'data'/'raw'/'01_raw_tickets.csv')
raw.head()

In [ ]:
raw.tail(); raw.info()
raw.describe(include='all').T.head(20)

In [ ]:
quality=pd.DataFrame({'missing':raw.isnull().sum(),'dtype':raw.dtypes.astype(str),'unique':raw.nunique(dropna=False)}).sort_values('missing',ascending=False)
quality.head(15)

In [ ]:
duplicate_tickets=raw.loc[raw.duplicated(subset=['Ticket_ID'],keep=False)].sort_values('Ticket_ID')
duplicate_tickets.iloc[:10,:8], len(duplicate_tickets)

In [ ]:
df=raw.drop_duplicates(subset=['Ticket_ID'],keep='first').copy()
for c in ['Employee_Name','Department','Location','Category','Subcategory','Priority','Status','Assigned_Technician','Channel']:
    df.loc[:,c]=df[c].astype('string').str.strip()
priority_map={'critical':'Critical','high':'High','medium':'Medium','med':'Medium','low':'Low'}
df['Priority']=df['Priority'].str.lower().map(priority_map)

In [ ]:
def parse_date(s):
    a=pd.to_datetime(s,format='%Y-%m-%d',errors='coerce')
    bad=a.isna() & s.notna()
    a.loc[bad]=pd.to_datetime(s.loc[bad],format='%d/%m/%Y',errors='coerce')
    return a
for c in ['Created_Date','Resolved_Date','Closed_Date']:
    df[c]=parse_date(df[c])
df['Created_Timestamp']=pd.to_datetime(df['Created_Date'].dt.strftime('%Y-%m-%d')+' '+df['Created_Time'],errors='coerce')

In [ ]:
# Business-aware technician imputation with groupby + transform-style logic.
pair_mode=(df.dropna(subset=['Assigned_Technician']).groupby(['Category','Location'])['Assigned_Technician']
           .agg(lambda x:x.mode().iloc[0] if not x.mode().empty else x.iloc[0]))
for idx in df.index[df['Assigned_Technician'].isna()]:
    key=(df.at[idx,'Category'],df.at[idx,'Location'])
    if key in pair_mode.index:
        df.at[idx,'Assigned_Technician']=pair_mode.loc[key]

In [ ]:
sla=pd.read_csv(ROOT/'data'/'processed'/'sla_policies.csv')
sla_map=sla.set_index('Priority')['SLA_Target_Hours'].to_dict()
df['SLA_Target_Hours']=df['Priority'].map(sla_map)
df['Resolution_Time_Hours']=pd.to_numeric(df['Resolution_Time_Hours'],errors='coerce')
df['SLA_Status_Calc']=np.where(df['Resolution_Time_Hours'].notna(),np.where(df['Resolution_Time_Hours']<=df['SLA_Target_Hours'],'Met','Breached'),'Pending')
df[['Priority','Resolution_Time_Hours','SLA_Target_Hours','SLA_Status_Calc']].head()

In [ ]:
# NumPy classification and outlier detection
r=df['Resolution_Time_Hours']
df['Resolution_Band']=np.select([r.le(2),r.le(4),r.le(8),r.le(24),r.le(72),r.le(168)],['0–2 Hours','2–4 Hours','4–8 Hours','8–24 Hours','1–3 Days','3–7 Days'],default='7+ Days')
vals=r.dropna(); q1,q3=np.percentile(vals,[25,75]); upper=q3+1.5*(q3-q1)
df['Resolution_Outlier']=np.where(r>upper,'Review','Typical')
df['Resolution_Outlier'].value_counts()

In [ ]:
# merge, concat, pivot_table and map support real quality checks
technicians=pd.read_csv(ROOT/'data'/'processed'/'technicians.csv')
merged=df.merge(technicians[['Technician_Name','Team']],left_on='Assigned_Technician',right_on='Technician_Name',how='left')
pivot=pd.pivot_table(merged,index='Category',columns='Priority',values='Ticket_ID',aggfunc='count',fill_value=0)
before=pd.DataFrame({'Stage':'Before','Missing':raw.isnull().sum()}).reset_index(names='Column')
after=pd.DataFrame({'Stage':'After','Missing':df.isnull().sum()}).reset_index(names='Column')
quality_compare=pd.concat([before,after],ignore_index=True)
pivot

In [ ]:
# dropna is used only where the analysis mathematically requires an observed resolution time.
resolved_only=df.dropna(subset=['Resolution_Time_Hours'])
resolved_only[['Resolution_Time_Hours','First_Response_Minutes']].describe()